# ☁️ Maquina-v7 — Cloud PC Linux + KDE Plasma + XRDP + Tailscale en Google Colab

> Escritorio remoto **GNU/Linux (KDE Plasma)** con **XRDP** (protocolo RDP nativo)
> y conexión segura por **Tailscale**. Sin necesidad de GPU para el escritorio.
> Funciona en cualquier cliente RDP (Microsoft Remote Desktop, aRDP, Remmina).

**Pasos:** ejecuta las celdas en orden: 1) Configuración, 2) Keep-Alive, 3) Instalar XRDP,
4) Estado. Opcional: Steam (5) y AnyDesk (6).

In [ ]:
#@title ⚙️ Configuración
GITHUB_USER = "jephersonRD"   #@param {type:"string"}
REPO        = "Maquina-v7"    #@param {type:"string"}

USERNAME = "jeph"             #@param {type:"string"}
PASSWORD = "medina"           #@param {type:"string"}
RESOLUTION = "1920x1080"      #@param {type:"string"}

# Tailscale: transporte TCP seguro para RDP (puerto 3389)
TAILSCALE_AUTHKEY = ""       #@param {type:"string"}  (pega tu authkey de https://login.tailscale.com/admin/settings/keys)
INSTALL_STEAM = False         #@param {type:"boolean"}

print("✅ Configuración lista. XRDP escucha en puerto 3389.")
print("   Conéctate con cualquier cliente RDP vía Tailscale.")

In [ ]:
#@title 🔒 Keep-Alive (anti-apagado de Colab)
from IPython.display import display, HTML
import threading, time

# Audio en silencio en loop (autocontenido, sin descargas externas)
AUDIO = "data:audio/wav;base64,UklGRqQ+AABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACABAAZGF0YYA+AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA... (line truncated to 2000 chars)
display(HTML(
  '<b>🔊 Mantén este audio en reproducción para evitar que Colab cierre la sesión.</b><br/>'
  f'<audio autoplay src="{AUDIO}" loop controls></audio>'
))

def _heartbeat():
    while True:
        time.sleep(60)
        print("♥ keep-alive", time.strftime('%H:%M:%S'))
threading.Thread(target=_heartbeat, daemon=True).start()
print('✅ Keep-alive activado. NO ocultes/cierres esta pestaña.')

In [ ]:
#@title Instalar escritorio KDE Plasma + XRDP
import os
if not os.path.isdir('/content/Maquina-v7'):
    tar_url = f"https://codeload.github.com/{GITHUB_USER}/{REPO}/tar.gz/refs/heads/main"
    !curl -fsSL {tar_url} -o /tmp/maquina-v7.tar.gz
    !tar xzf /tmp/maquina-v7.tar.gz -C /content
    !mv /content/{REPO}-main /content/Maquina-v7
    !rm -f /tmp/maquina-v7.tar.gz
%cd /content/Maquina-v7/scripts
!bash setup_xrdp.sh {USERNAME} {PASSWORD} {RESOLUTION}


In [ ]:
#@title 🌐 Tailscale (conexión segura vía RDP)
#@markdown Conecta Tailscale para acceder al escritorio de forma segura desde cualquier dispositivo.
import subprocess
if TAILSCALE_AUTHKEY:
    %cd /content/Maquina-v7/scripts
    !bash setup_tailscale.sh "{TAILSCALE_AUTHKEY}"
    ip = subprocess.run("tailscale ip -4 2>/dev/null | head -n1", shell=True, capture_output=True, text=True).stdout.strip()
    if ip:
        print("\n🌐 IP de Tailscale:", ip)
        print("   Conéctate por RDP a:", ip + ":3389")
    else:
        print("\n⚠️ Tailscale no obtuvo IP. Verifica la authkey.")
else:
    print("⚠️ TAILSCALE_AUTHKEY vacío. Agrega tu authkey para conectar por Tailscale.")
    print("   Sin Tailscale, solo puedes conectar por IP local (si está disponible).")

In [ ]:
#@title Estado y datos de conexión
import subprocess
print('🔌 Servidor RDP: puerto 3389')
listening = subprocess.run("ss -ltn 2>/dev/null | grep -q ':3389'", shell=True).returncode == 0
print('Puerto 3389  :', '✅ escuchando' if listening else '❌ NO escucha (revisa servicios XRDP)')
ip = subprocess.run("tailscale ip -4 2>/dev/null | head -n1", shell=True, capture_output=True, text=True).stdout.strip()
if ip:
    print('\n🌐 IP Tailscale:', ip)
    print('   Conéctate por RDP a:', ip + ':3389')
print('\n📱 Clientes RDP compatibles:')
print('   - Microsoft Remote Desktop (Android/Windows)')
print('   - aRDP (Android)')
print('   - Remmina (Linux)')
print('   - xfreerdp (Linux): xfreerdp /v:' + (ip or 'IP_TAILSCALE') + ':3389 /u:{USERNAME} /p:{PASSWORD}')

In [ ]:
#@title 🎮 Instalar Steam (opcional)
if INSTALL_STEAM:
    %cd /content/Maquina-v7/scripts
    !bash install_steam.sh
else:
    print('Omitido (INSTALL_STEAM = False).')

In [ ]:
#@title Instalar AnyDesk (opcional, celda separada)
%cd /content/Maquina-v7/scripts
!bash setup_anydesk.sh